In [0]:
import requests
from pyspark.sql.functions import current_timestamp

# 1. Define the API Endpoints for Pune
weather_url = "https://api.open-meteo.com/v1/forecast?latitude=18.5204&longitude=73.8567&current=temperature_2m,wind_speed_10m"
aqi_url = "https://air-quality-api.open-meteo.com/v1/air-quality?latitude=18.5204&longitude=73.8567&current=pm10,pm2_5,carbon_monoxide"

# 2. Fetch the Data
weather_response = requests.get(weather_url).json()
aqi_response = requests.get(aqi_url).json()

# 3. Convert JSON to Spark DataFrames
# We wrap the dictionary in a list [ ] so Spark knows it represents a single row of data
weather_df = spark.createDataFrame([weather_response])
aqi_df = spark.createDataFrame([aqi_response])

# 4. Add Ingestion Timestamps (Crucial for tracking when data arrived)
bronze_weather_df = weather_df.withColumn("ingestion_timestamp", current_timestamp())
bronze_aqi_df = aqi_df.withColumn("ingestion_timestamp", current_timestamp())

# 5. Write Data to Delta Tables in the Bronze Layer
# mode("append") ensures we just add new data every time this runs, without deleting old data
bronze_weather_df.write.format("delta").mode("append").saveAsTable("bronze_pune_weather")
bronze_aqi_df.write.format("delta").mode("append").saveAsTable("bronze_pune_aqi")

print("Data successfully landed in the Bronze Layer!")

Data successfully landed in the Bronze Layer!


In [0]:
%sql
SELECT * FROM bronze_pune_weather;

current,current_units,elevation,generationtime_ms,latitude,longitude,timezone,timezone_abbreviation,utc_offset_seconds,ingestion_timestamp
"Map(time -> 2026-02-28T08:30, interval -> 900, temperature_2m -> 32.1, wind_speed_10m -> 14.8)","Map(time -> iso8601, interval -> seconds, temperature_2m -> °C, wind_speed_10m -> km/h)",561.0,0.05125999450683594,18.5,73.875,GMT,GMT,0,2026-02-28T08:35:02.919Z
"Map(time -> 2026-02-28T08:30, interval -> 900, temperature_2m -> 32.1, wind_speed_10m -> 14.8)","Map(time -> iso8601, interval -> seconds, temperature_2m -> °C, wind_speed_10m -> km/h)",561.0,0.043272972106933594,18.5,73.875,GMT,GMT,0,2026-02-28T08:35:30.928Z


In [0]:
from pyspark.sql.functions import col, to_timestamp

# 1. Read the raw data from your Bronze tables
bronze_weather_df = spark.read.table("bronze_pune_weather")
bronze_aqi_df = spark.read.table("bronze_pune_aqi")

# 2. Transform the Weather Data
silver_weather_df = bronze_weather_df.select(
    # Extract time and convert to proper Spark timestamp
    to_timestamp(col("current.time"), "yyyy-MM-dd'T'HH:mm").alias("observation_time"),
    # Extract metrics and cast as floats (decimals)
    col("current.temperature_2m").cast("float").alias("temperature_celsius"),
    col("current.wind_speed_10m").cast("float").alias("wind_speed_kmh"),
    # Keep the ingestion timestamp for auditing
    col("ingestion_timestamp")
)

# 3. Transform the AQI Data
silver_aqi_df = bronze_aqi_df.select(
    to_timestamp(col("current.time"), "yyyy-MM-dd'T'HH:mm").alias("observation_time"),
    col("current.pm10").cast("float").alias("pm10"),
    col("current.pm2_5").cast("float").alias("pm2_5"),
    col("current.carbon_monoxide").cast("float").alias("carbon_monoxide"),
    col("ingestion_timestamp")
)

# 4. Write the clean data to Silver Delta Tables
silver_weather_df.write.format("delta").mode("append").saveAsTable("silver_pune_weather")
silver_aqi_df.write.format("delta").mode("append").saveAsTable("silver_pune_aqi")

print("Data successfully cleaned and landed in the Silver Layer!")

Data successfully cleaned and landed in the Silver Layer!


In [0]:
%sql
SELECT * FROM silver_pune_weather;


observation_time,temperature_celsius,wind_speed_kmh,ingestion_timestamp
2026-02-28T08:30:00.000Z,32.1,14.8,2026-02-28T08:35:02.919Z
2026-02-28T08:30:00.000Z,32.1,14.8,2026-02-28T08:35:30.928Z


In [0]:
from pyspark.sql.functions import col, when, date_trunc

# 1. Read the clean data from your Silver tables
silver_weather_df = spark.read.table("silver_pune_weather")
silver_aqi_df = spark.read.table("silver_pune_aqi")

# 2. FIX: Round the observation times down to the nearest hour so they match
weather_hourly = silver_weather_df.withColumn("join_time", date_trunc("hour", col("observation_time")))
aqi_hourly = silver_aqi_df.withColumn("join_time", date_trunc("hour", col("observation_time")))

# 3. Join the tables together based on our new rounded 'join_time'
joined_df = weather_hourly.join(
    aqi_hourly.drop("ingestion_timestamp", "observation_time"), 
    on="join_time", 
    how="inner"
)

# 4. Add Business Logic: Categorize Air Quality
gold_df = joined_df.withColumn(
    "air_quality_category",
    when(col("pm2_5") <= 12.0, "Good")
    .when((col("pm2_5") > 12.0) & (col("pm2_5") <= 35.4), "Moderate")
    .when((col("pm2_5") > 35.4) & (col("pm2_5") <= 55.4), "Unhealthy for Sensitive Groups")
    .otherwise("Unhealthy")
).drop("join_time") # Clean up our temporary join column

# 5. Write to Gold Delta Table
gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_pune_metrics")

print("Gold table updated with rounded timestamps!")

Gold table updated with rounded timestamps!


In [0]:

%sql
SELECT * FROM gold_pune_metrics ORDER BY observation_time DESC;

observation_time,temperature_celsius,wind_speed_kmh,ingestion_timestamp,pm10,pm2_5,carbon_monoxide,air_quality_category
2026-02-28T08:30:00.000Z,32.1,14.8,2026-02-28T08:35:02.919Z,14.7,12.7,257.0,Moderate
2026-02-28T08:30:00.000Z,32.1,14.8,2026-02-28T08:35:30.928Z,14.7,12.7,257.0,Moderate
2026-02-28T08:30:00.000Z,32.1,14.8,2026-02-28T08:35:02.919Z,14.7,12.7,257.0,Moderate
2026-02-28T08:30:00.000Z,32.1,14.8,2026-02-28T08:35:30.928Z,14.7,12.7,257.0,Moderate
